# 2-3 Streamlit을 활용한 RAG 배포

원본 슬라이드: [2-3-streamlit-rag.pdf](./2-3-streamlit-rag.pdf)  
이 노트북은 슬라이드의 개념 순서를 따라가는 학습용 해설과 작은 실험입니다. 원본 PDF는 그대로 보존합니다.

## 학습 목표
- Streamlit의 재실행 모델을 이해한다.
- title·caption·cache_resource·session_state의 쓰임을 설명한다.
- 채팅 입력·출력·스트리밍을 RAG 답변 흐름과 연결한다.

**실습 방식:** 예제를 먼저 실행해 관찰하고, CodeMate에 개선을 요청한 뒤 생성 결과를 직접 검토합니다. 실제 CodeMate 메뉴·버튼 이름은 강의장 환경에서 확인하세요.


## 슬라이드별 찾아보기

각 쪽의 소제목과 첫 핵심 문구를 찾아가기 위한 표입니다. 본문의 설명·실습과 함께 읽고, 원본의 그림·도표·화면 예시는 쪽 번호를 눌러 확인하세요. PDF에서 수식 또는 그림 안 글자가 추출되지 않는 쪽은 원본 화면을 기준으로 읽습니다.

| PDF 쪽 | 소제목 | 확인할 내용 |
|---|---|---|
| [1쪽](./2-3-streamlit-rag.pdf#page=1) | 2-3 | Streamlit을 활용한 RAG 배포 |
| [2쪽](./2-3-streamlit-rag.pdf#page=2) | 학습 목차 | • Streamlit |
| [3쪽](./2-3-streamlit-rag.pdf#page=3) | Streamlit | 원본 슬라이드의 그림·배치 확인 |
| [4쪽](./2-3-streamlit-rag.pdf#page=4) | Streamlit 소개 | Streamlit: Python 코드만으로 빠르게 웹 애플리케이션을 만들 수 있게 해주는 프레임워크 |
| [5쪽](./2-3-streamlit-rag.pdf#page=5) | Python 코드 기반 웹 UI | Streamlit 앱은 위에서 아래로 실행되는 Python 스크립트처럼 작성 |
| [6쪽](./2-3-streamlit-rag.pdf#page=6) | Streamlit 앱 기본 구성 | 원본 슬라이드의 그림·배치 확인 |
| [7쪽](./2-3-streamlit-rag.pdf#page=7) | title과 caption (1) | st.title : 앱의 가장 큰 제목을 화면에 표시 |
| [8쪽](./2-3-streamlit-rag.pdf#page=8) | title과 caption (2) | st.caption: 제목 아래에 짧은 보조 설명을 표시 |
| [9쪽](./2-3-streamlit-rag.pdf#page=9) | cache_resource | @st.cache_resource : 무거운 리소스를 한 번만 만들고 재사용하기 위한 데코레이터 |
| [10쪽](./2-3-streamlit-rag.pdf#page=10) | 채팅 인터페이스 구성 | 원본 슬라이드의 그림·배치 확인 |
| [11쪽](./2-3-streamlit-rag.pdf#page=11) | session_state | st.session_state : 사용자 세션 동안 유지해야 하는 값을 저장하는 공간 |
| [12쪽](./2-3-streamlit-rag.pdf#page=12) | chat_message 와 chat_input | st.chat_message : 채팅 메시지를 역할별 말풍선 형태로 표시 |
| [13쪽](./2-3-streamlit-rag.pdf#page=13) | write_stream | st.write_stream : 생성되는 텍스트를 한 번에 보여주지 않고 순차적으로 화면에 출력 |


## 1. Streamlit 앱의 실행 방식 (슬라이드 3–5)

Streamlit은 Python 스크립트로 웹 UI를 만듭니다. `streamlit run app.py`로 실행하고, 입력이 발생하면 스크립트가 위에서 아래로 **다시 실행**될 수 있습니다. 유지할 대화는 `st.session_state`에, 초기화 비용이 큰 모델·저장소는 적절한 캐시에 둡니다. 노트북 안에서 Streamlit 화면을 직접 띄우는 대신 앱 구조를 읽고 검토합니다.

## 2. 화면의 첫 정보: title과 caption (슬라이드 6–8)

`st.title`은 앱 이름, `st.caption`은 어떤 자료를 근거로 답하는지 짧게 설명합니다. 원본 슬라이드는 “보험 약관 QA 챗봇”과 “메리츠 펫보험 약관”을 예로 듭니다. 앱 문구와 실제 로드한 문서가 다르면 사용자를 오도하므로 둘을 맞춰야 합니다.

```python
import streamlit as st
st.title("보험 약관 QA 챗봇")
st.caption("메리츠 펫보험 약관을 기반으로 답변합니다.")
```

## 3. 리소스 캐시 (슬라이드 9)

`@st.cache_resource`는 모델·벡터 저장소 등 무거운 리소스를 재사용할 때 씁니다. 원본의 `load_rag_chain()` 예는 PDF 로드→벡터 저장소→RAG 체인 구성을 한 번 묶습니다. 문서가 바뀌었는데 캐시가 낡은 결과를 돌려주지 않도록 갱신 조건을 설계하세요. 강의장에서는 실제 문서 경로와 모델 연결을 확인해야 합니다.

```python
@st.cache_resource
def load_rag_chain():
    # PDF 로드, 분할, 임베딩, 검색기와 체인 연결
    ...
```

## 4. 대화 기록: session_state (슬라이드 10–11)

재실행 뒤에도 메시지를 보여주려면 목록을 `st.session_state`에 보관하고 매번 다시 그립니다. 다음 순수 Python 예제는 상태가 재실행 동안 유지되는 원리만 모의합니다.

In [ ]:
session_state = {}
def render_cycle(state, new_message=None):
    state.setdefault("messages", [])
    if new_message:
        state["messages"].append(new_message)
    return [(m["role"], m["content"]) for m in state["messages"]]
print(render_cycle(session_state, {"role": "user", "content": "보장 범위는?"}))
print(render_cycle(session_state))  # 재실행을 가정해도 기록이 남음

## 5. chat_message, chat_input, write_stream (슬라이드 12–13)

`st.chat_input`은 질문을 받고, `st.chat_message`는 역할별 말풍선을 표시합니다. `st.write_stream`은 답변 조각을 화면에 순차 표시합니다. 스트리밍이 끝난 **완성된 답변**을 대화 기록에 저장해야 다음 재실행에서 같은 내용을 다시 보여줄 수 있습니다.

```python
if prompt := st.chat_input("질문을 입력하세요"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("assistant"):
        response = st.write_stream(load_rag_chain().stream(prompt))
    st.session_state.messages.append({"role": "assistant", "content": response})
```

In [ ]:
def answer_chunks():
    for part in ["검색 근거에 따르면 ", "해당 내용은 ", "문서 2쪽에 있습니다."]:
        yield part
complete_answer = "".join(answer_chunks())
session_state["messages"].append({"role": "assistant", "content": complete_answer})
print(render_cycle(session_state))

**검토:** 실제 RAG 체인의 `stream(prompt)` 결과가 문자열 조각인지, 메시지 객체인지 확인해야 합니다. `st.write_stream`이 반환한 값과 `session_state`에 저장할 형식도 실제 실행으로 확인하세요. 출처가 없는 답변은 완성된 답변처럼 표시하지 마세요.

### CodeMate 페어 실습

> 원본 슬라이드 7–13의 흐름을 사용해 Streamlit RAG 앱의 최소 구조를 제안해 주세요. 먼저 필요한 입력·출력과 재실행 시 유지할 상태를 적고, 검색 근거가 없으면 “문서에서 확인할 수 없습니다”라고 보여 주세요. `load_rag_chain()`의 실제 구현은 프로젝트의 기존 RAG 실습을 확인한 뒤 연결하고, 없는 파일이나 비밀키를 임의로 만들지 마세요.

**리뷰 질문:** 앱 제목과 실제 문서가 일치하나요? 입력 후 대화가 중복되거나 사라지지 않나요? 캐시 갱신 조건과 출처 표기가 있나요?

## 정리

1. 일반 변수 대신 `session_state`가 필요한 이유는 무엇인가요?
2. 어떤 리소스에 `cache_resource`를 쓰나요?
3. 스트리밍 응답을 기록할 때 무엇을 저장해야 하나요?